# STAT 301 Project — Online Shoppers Purchasing Intention

**Name:** Samika Dewan  
**Group:**  Group-25   
**Student number:** 27276179 

---


## Section 1: Data Description


### 1. Descriptive Summary

#### Data description

This data contains information about user behavior when online shopping. Each row in the dataset represents one online shopping session, and the variables describe various aspects of user behavior, technical attributes, and timing information. The dataset includes both quantitative and categorical variables, such as the number and duration of pages visited, bounce and exit rates, month of visit, type of operating system, and whether the session occurred on a weekend.  

- **Number of observations:** 12330  
- **Number of variables:** 18
#### Variables description
  | Variable Name | Type | Description |
|-----------------------------|------------------|-------------------------------------------------------------|
| Administrative |Numeric(Integer) | Number of administrative pages visited by the user. |
| Administrative_Duration | Numeric(Integer) | Total time spent on administrative-related pages (in seconds). |
| Informational | Numeric(Integer) | Number of informational pages visited by the user. |
| Informational_Duration | Numeric(Integer) | Total time spent on informational pages (in seconds). |
| ProductRelated | Numeric(Integer) | Number of product-related pages visited by the user. |
| ProductRelated_Duration | Numeric(Continuous) | Total time spent on product-related pages (in seconds). |
| BounceRates | Numeric(Continuous) | percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. |
| ExitRates | Numeric(Continuous) | percentage of this page being the last session. |
| PageValues | Numeric(Integer) | Average value for a web page that a user visited before completing an e-commerce transaction.  |
| SpecialDay | Numeric(Integer) | Closeness of the site visit date to a special day (e.g., Mother’s Day, Christmas). Ranges from 0 to 1. |
| Month | Categorical | Month of the visit (Feb–Dec). |
| OperatingSystems | Categorical | Type of operating system used by the visitor (e.g., Windows, Mac). |
| Browser | Categorical | Type of browser used (e.g., Chrome, Firefox, Safari). |
| Region | Categorical | Geographic region of the visitor. |
| TrafficType | Categorical | Type of traffic source (e.g., direct, referral, search). |
| VisitorType | Categorical | Indicates if the visitor is a Returning Visitor or New Visitor. |
| Weekend | Binary (True/False) | Indicates whether the visit occurred on a weekend. |
| Revenue | Binary (0/1) | Indicates whether the session ended with a purchase (1 = purchase, 0 = no purchase). |

---



### 2. Source and Information 
This dataset is the **Online Shoppers Purchasing Intention** dataset published by UCI Machine Learning Repository. 
It contains session-level features for e-commerce website visits and a binary outcome `Revenue` indicating whether a purchase occurred. 



### 3. Pre-selection of Variables 
Initially, I am considering a fairly rich set of explanatory variables that describe different aspects of each session. Below I have divided them in categories that somewhat measure similar aspects and may be potential covariates. In later stages, I will identify a smaller set of non-redundant explanatory variables.

- **Engagement / browsing depth**
  - `Administrative`, `Informational`, `ProductRelated`: number of pages of each type viewed.
  - `Administrative_Duration`, `Informational_Duration`, `ProductRelated_Duration`:
    total time spent on each type of page.
  - `PageValues`: an estimate of the value of the pages visited.


- **Exit behaviour**
  - `BounceRates`: fraction of visits in which the user leaves after viewing only one page.
  - `ExitRates`: fraction of pageviews in which the user exits the site from that page.


- **Timing**
  - `SpecialDay`: a number between 0 and 1 indicating how close the visit is to a special
    shopping day (e.g., Valentine’s Day). Larger values mean the session occurred closer
    to such a day.
  - `Month` and `Weekend` (used mainly for EDA to explore seasonality and weekday vs weekend patterns).


- **Visitor profile**
  - `VisitorType`: whether the visitor is new, returning, or other.

## Section 2: Scientific Question


### 1. Scientific Question 
How do different aspects of a user’s browsing behaviour and visit timing — such as how many pages they view, how long they stay, how often they exit pages, and how close the visit is to a special shopping day — associate with the likelihood that an online shopping session ends with a purchase (Revenue)?

### 2. Name the Response 
**Response:** `Revenue` (binary: 1 if the session resulted in a purchase, 0 otherwise).

### 3. Aim: Prediction, Inference, or Both? 
It is an **Inference** model, because we aim to understand whether session behavior — including BounceRates, ExitRates, and Administrative activities — is associated with the purchase outcome (`Revenue`), rather than predicting it.

## Section 3: Exploratory Data Analysis and Visualization (EDA)

In [ ]:
library(tidyverse)
library(repr)
library(infer)
library(cowplot)
library(broom)
library(car)

In [ ]:
online_shopping_raw <- read.csv("online_shoppers_intention.csv")

online_shopping <- online_shopping_raw %>%
filter(is.na(Region) | Region != 1L)

head(online_shopping)
nrow(online_shopping)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)

online <- read_csv("online_shoppers_intention.csv") %>%
  filter(is.na(Region) | Region != 1) %>%
  mutate(
    # Response as factor with clear labels
    Revenue = factor(
      Revenue,
      levels = c(FALSE, TRUE),
      labels = c("No Purchase", "Purchase")
    ),
    # Factor for near vs regular special days
    NearSpecialDay = if_else(
      SpecialDay > 0,
      "Near special shopping day",
      "Regular day"
    ),
    # More readable weekend factor
    Weekend = if_else(
      Weekend,
      "Weekend",
      "Weekday"
    ),
    # Simplify visitor types (top 2 levels + Other)
    VisitorType_simple = fct_lump_n(VisitorType, n = 2, other_level = "Other")
  )

In [ ]:
x_limit <- quantile(online$ProductRelated, 0.99, na.rm = TRUE)
y_limit <- quantile(online$ExitRates, 0.99, na.rm = TRUE)

eda_plot <- ggplot(online,
                   aes(x = ProductRelated, y = ExitRates, colour = Revenue)) +
            
  geom_point(alpha = 0.25, size = 1) +
  
  geom_smooth(
    method = "loess",
    se = FALSE,
    linewidth = 1) +
  
  facet_grid(
    rows = vars(NearSpecialDay),
    cols = vars(VisitorType_simple)) +
  
  coord_cartesian(
    xlim = c(0, x_limit),
    ylim = c(0, y_limit)) +
  
  labs(
    title = "Product Exploration vs Exit Rates by Purchase Outcome",
    subtitle = "Faceted by special shopping days and visitor type",
    x = "Number of product-related pages viewed (ProductRelated)",
    y = "Average exit rate from visited pages (ExitRates)",
    colour = "Purchase outcome") +
  
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(size = 11),
    strip.text = element_text(face = "bold"),
    legend.position = "bottom",
    panel.grid.minor = element_blank()) +
  
  guides(
    colour = guide_legend(
      override.aes = list(alpha = 1, size = 3)))

eda_plot

### Interpretations

- **Why this plot?**  
  This visualization focuses on two key behavioural variables, `ProductRelated` (how many product pages are viewed) and `ExitRates` (how often users exit from pages), and relates them to the binary response `Revenue` (Purchase vs No Purchase). The points are coloured by purchase outcome and we facet by both `NearSpecialDay` (rows) and a simplified `VisitorType` (columns), so each panel shows a specific combination of timing and visitor type. This allows us to compare browsing and exit behaviour between purchasers and non-purchasers across different types of sessions in a single figure.

- **Brief results:**  
  Across almost all panels, purchase sessions tend to cluster where users view **more product-related pages** and have **lower exit rates**, while non-purchase sessions are concentrated at low `ProductRelated` and higher `ExitRates`. The smooth curves show a generally downward trend for purchases: as users view more product pages, their exit rates tend to decrease. Near special shopping days, this separation between purchase and non-purchase sessions is even more pronounced, especially for returning visitors.

- **What we learn / potential issues:**  
  The plot suggests that deeper product exploration and lower exit rates are strongly associated with purchases, and that this pattern is amplified near special shopping days and among returning visitors. However, some panels (e.g., “Other” visitors or very high page counts) appear to have fewer points, so the smooth curves there may be driven by relatively small sample sizes. This motivates using a logistic regression model with carefully selected, non-redundant predictors to quantify these associations while being cautious about over-interpreting sparse regions of the plot.

## Section 4: Method and Plan

### 1. Step-by-step plan
1. Identify all candidate numeric predictors.
2. Fit an initial logistic regression using all numeric predictors and compute a correlation matrix.  
3. Select numeric explanatory variables using the coefficient patterns, p-values, and multicollinearity diagnostics. 
4. Rebuild the logistic regression model using the cleaned set of numeric predictors and then include all categorical variables.
5. Check VIFs for the combined model (numeric + categorical).
6. Interpret and select the final explanatory variable set for modelling.

### 2. Justification
This methodology identifies the best explanatory variables by first checking numeric predictors for redundancy using a full model, correlation matrix, and VIFs. Removing highly correlated variables prevents multicollinearity and keeps only predictors that add unique information. After selecting the numeric set, adding categorical variables and confirming their GVIF values ensures they do not introduce instability. This systematic process produces a final variable set that is statistically reliable and holistically interpretable for logistic regression.

### 3. Assumptions
- The logistic regression assumes a linear relationship between each predictor and the log-odds of purchase.  
- Observations are treated as independent sessions, meaning one user's behaviour does not influence another’s.  
- No strong multicollinearity is assumed among predictors; will be validated through correlation analysis and VIFs.  

### 4. Limitations
- The dataset is observational, so all results describe associations rather than causal effects.  
- Although multicollinearity was addressed, remaining predictors may still interact in ways not modelled here.

## Section 5: Code and Outcome

#### Step 0: Cleaning and prepping data

In [ ]:
online <- read_csv("online_shoppers_intention.csv") %>%
  filter(is.na(Region) | Region != 1) %>%
  mutate(
    Revenue = factor(
      Revenue,
      levels = c(FALSE, TRUE),
      labels = c("No Purchase", "Purchase")),
    Weekend = factor(Weekend, levels = c(FALSE, TRUE),
                     labels = c("Weekday", "Weekend")),
    VisitorType_simple = forcats::fct_lump_n(VisitorType, n = 2, other_level = "Other"),
    VisitorType_simple = forcats::fct_relevel(VisitorType_simple, "Returning_Visitor"),
    Month = factor(Month))

#### Step 1: Defining all numeric predictors and fitting all to a logistic model

In [ ]:
cand_num_vars <- c("Administrative", "Administrative_Duration","Informational", "Informational_Duration","ProductRelated", 
                   "ProductRelated_Duration","BounceRates", "ExitRates","PageValues", "SpecialDay")

#### Step 2: Fit an initial logistic regression using all numeric predictors and compute a correlation matrix.

In [ ]:
full_formula_num <- as.formula(
  paste("Revenue ~", paste(num_vars, collapse = " + "))
)

# Fitting the full numeric logistic model
full_glm_num <- glm(
  full_formula_num,
  data = online,
  family = "binomial"
)

summary(full_glm_num)

# VIFs for the full numeric model
full_vif_num <- vif(full_glm_num)
full_vif_num

corr_matrix <- online %>%
  select(all_of(num_vars)) %>%
  cor()

corr_matrix

#### Step 3: Select numeric explanatory variables using the coefficient patterns, p-values, and multicollinearity diagnostics. 

The correlation matrix showed that each page‐count variable was extremely highly correlated with its corresponding duration measure (e.g., **ProductRelated vs. ProductRelated_Duration: r ≈ 0.86**). Since these pairs represent the same underlying behaviour, I kept the page‐count variables (**Administrative, Informational, ProductRelated**) and removed their duration counterparts to avoid redundancy. A moderate correlation was also found between **BounceRates** and **ExitRates** (**r ≈ 0.31**). Because *ExitRates* provides a broader measure of page exits, it was retained while *BounceRates* was removed. Variables such as **PageValues** and **SpecialDay** showed low correlations with other predictors and had VIFs near 1, indicating they provide unique explanatory information.

After removing redundant predictors, the final numeric set is:`Administrative`, `Informational`, `ProductRelated`, `ExitRates`, `PageValues`, and `SpecialDay` all with VIF < 5.


#### Step 4: Rebuild the logistic regression model using the cleaned set of numeric predictors and then include all categorical variables.
#### Step 5: Check VIFs for the combined model (numeric + categorical). 

In [ ]:
# Numeric variables chosen in Step 3
best_num_vars <- c("Administrative","Informational","ProductRelated","ExitRates","PageValues","SpecialDay")

# Adding categorical variables
cat_vars <- c("Weekend", "VisitorType_simple")   

# Re-building final model formula: Revenue ~ numerics + categoricals
final_formula <- as.formula(paste("Revenue ~",paste(c(best_num_vars, cat_vars), collapse = " + ")))

best_glm <- glm(
  final_formula,
  data = online,
  family = "binomial")

summary(best_glm)

best_vif <- vif(best_glm)
best_vif

#### Final Interpretation
- `PageValues` has a **positive** coefficient with a very small p-value (**p < 0.001**), indicating that sessions visiting higher-value pages are much more likely to result in a purchase.

- `ExitRates` has a **negative** coefficient with a very small p-value (**p < 0.001**), so sessions where users exit pages more frequently are much less likely to convert.

- `SpecialDay` has a statistically significant coefficient (**p < 0.05**), showing that timing relative to special shopping days is meaningfully related to purchase probability after adjusting for other variables.

- The remaining predictors (`Administrative`, `Informational`, `Weekend`, `VisitorType_simple`) have small coefficients with large p-values (**p > 0.05**), suggesting they contribute little additional explanatory power once the main predictors are in the model.

- Therefore, the response variable is `Revenue` and the final explanatory variables are `Administrative`, `Informational`, `ProductRelated`,`ExitRates`,`PageValues`,`SpecialDay`,`Weekend`, `VisitorType_simple`.